> Processing of the results from training (including *clusterization*)

In [ ]:
import pathlib

import numpy as np
import pandas as pd

from torch.utils.data import DataLoader
import hdbscan

import altair as alt
import plotly
import plotly.express as px
import plotly.graph_objects as go

import trapallada.data
import trapallada.util
import trapallada.core
import trapallada.plot
import trapallada.analysis

# import ipywidgets as widgets

So that `import`ed Python modules are automatically reloaded when modified

In [ ]:
%load_ext autoreload
%autoreload 2

So that *plotly* pictures can be exported to HTML

In [ ]:
plotly.offline.init_notebook_mode()

The base directory

In [ ]:
base_dir = trapallada.analysis.get_base_directory()
base_dir

# Input Files

Widget to choose a *sub*directory with results

In [ ]:
search_dir = base_dir / 'results' / 'normalization' / 'normalized_rate'
assert search_dir.exists()
combo = trapallada.analysis.make_combo_widget(search_dir)
display(combo)

Particular *sub*directory with the trained model. It should be a `str` so that parameters can be injected with papermill.

In [ ]:
model_dir = (search_dir / combo.value).as_posix()
# ===============
# model_subdir = 'results/normalization/normalized_rate/vanilla_x_sd_015'
# model_subdir = 'results/normalization/normalized_rate/fft_vanilla_x_sd_00025'
# ===============
# model_subdir = 'results/normalization/clean/vanilla'
model_subdir = 'results/normalization/count_rate/vanilla_x_sd_015'
# ===============
model_subdir = "results/masked/vanilla/vanilla_x_sd_015"

Just in case

In [ ]:
model_dir = base_dir / model_subdir
# model_dir = pathlib.Path('/tmp') / 'normalization' / 'count_rate' / 'vanilla_x_sd_015'
display(model_dir)
assert model_dir.exists()

From the results directory,
- the `Dataset`,
- the headers of the corresponding files,
- the ML model,
- a `DataFrame` with results from training, 
- a `dict` with the parameters

In [ ]:
ds, positions_df, model, results_df, parameters = trapallada.analysis.load_results_dir(model_dir)

The directory with the full signals is inferred

In [ ]:
full_signals_dir = (pathlib.Path.home() / parameters['dataset']['files']['signals_file']).parent / 'full_signals'

Results from training

In [ ]:
results_df

# Training curves

In [ ]:
trapallada.analysis.metric_chart(results_df, 'loss') | \
    trapallada.analysis.metric_chart(results_df, 'loglikelihood') | \
        trapallada.analysis.metric_chart(results_df, 'kl')

Number of parameters in the model

In [ ]:
sum(trapallada.util.get_layers_n_parameters(model).values())

Data loaders for both the training and validation sets

In [ ]:
(train_dl, validation_dl), (train_ds, validation_ds) = trapallada.data.dataloaders_from_dataset(
    ds,
    splitting_parameters=parameters['training']['train_test_split'] | dict(random_state=parameters['PRNG seeds']['training-validation splitting']),
    batch_size=parameters['training']['batch_size'], batching_random_state=parameters['PRNG seeds']['training dataloader'], return_datasets=True)

A batch from the **validation** set

In [ ]:
# iter_dl = iter(train_dl) # training
iter_dl = iter(validation_dl) # validation
next(iter_dl), next(iter_dl)
batch = next(iter_dl)
batch[1].shape

What are the different labels, *if any* (useful to check whether every class is present)?

In [ ]:
batch[2].unique()

An element from the above batch is encoded, decoded, and a prediction plotted

In [ ]:
model.decoder.layers[-1]
model.encoder.layers[0]
model.encoder.layers_spatial_sizes

In [ ]:
i_s, *_ = batch
i = 31

trapallada.analysis.predict_and_plot(model, ds, i_s[i].item())

## Javi's paper

A function that looks up a particular source (a `FTZ` file) and plots the requested window along with its reconstruction. It returns a `Series` **relative to the full dataset**.

In [ ]:
def plot_source_window(source: str, i_window: int = -1):

    df = ds.metadata_df[ds.metadata_df['source'] == source]

    try:
        series = df.iloc[i_window]
    
    except IndexError:

        print('not found')

        return None
    
    index = series.name

    display(trapallada.analysis.predict_and_plot(model, ds, index))
    
    # print(f'seen during training? {index in train_ds[:][0]}' )

    return series

In [ ]:
javi_obs = plot_source_window('P0605950101PNS003SRCTSR8001.FTZ', -1)
javi_meta = [javi_obs]

In [ ]:
javi_obs = plot_source_window('P0605950101PNS003SRCTSR8003.FTZ')
javi_meta.append(javi_obs)

In [ ]:
javi_obs = plot_source_window('P0605950101PNS003SRCTSR8005.FTZ')
javi_meta.append(javi_obs)

In [ ]:
javi_obs = plot_source_window('P0605950101PNS003SRCTSR8007.FTZ')
javi_meta.append(javi_obs)

In [ ]:
i_javi = [e.name for e in javi_meta if e is not None]
i_javi

# Latent space projection

In [ ]:
# latents = trapallada.analysis.Latents(DataLoader(ds, batch_size=128, shuffle=False), model.encoder, parameters['model']['dim_z'])

In [ ]:
# latents.plot_ith(20)

## 2D

The **full** dataset is encoded into the latent space (last columns in the resulting `DataFrame`)

In [ ]:
parameters['dataset']

In [ ]:
fulldataset_dl = DataLoader(ds, batch_size=128, shuffle=False)
proj_df = trapallada.util.make_projections_df(ds, fulldataset_dl, model.encoder)
proj_df.loc[train_ds[:][0], 'set'] = 'training'
proj_df.loc[validation_ds[:][0], 'set'] = 'validation'
proj_df

Embedded space is close to a standard Gaussian?

In [ ]:
proj_df.filter(like='z_').agg(['mean', 'std'])

### Clustering

Clustering using [HDBSCAN](https://github.com/scikit-learn-contrib/hdbscan) with pretty much every default parameter. See [this notebook](https://nbviewer.org/github/scikit-learn-contrib/hdbscan/blob/master/notebooks/How%20HDBSCAN%20Works.ipynb) for an explanation.

In [ ]:
def columns_clusters(like: str, min_cluster_size: int) -> pd.Series:

    selected_cols = proj_df.filter(like=like)

    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size)
    
    labels = clusterer.fit_predict(selected_cols)
    
    print(f'number of clusters: {len(np.unique(labels))}')
    
    return pd.Categorical(labels)

Clusting based on the **full** latent space

In [ ]:
cluster_in_z = 'cluster in z'
proj_df[cluster_in_z] = columns_clusters('z_', 10)

The number of samples in every cluster

In [ ]:
n_samples_per_cluster = proj_df[cluster_in_z].value_counts()
n_samples_per_cluster

An array containing the indexes of the samples (signals) that do not belong to the large cluster (which is, presumably, noise).

In [ ]:
i_minority = proj_df[proj_df[cluster_in_z] != n_samples_per_cluster.idxmax()].index
len(i_minority)

Clusting based on the tSNE projection

In [ ]:
cluster_in_projected_space = 'cluster in projected space'
proj_df[cluster_in_projected_space] = columns_clusters('tsne_', 80)
proj_df[cluster_in_projected_space].value_counts()

### Extracting positions

Information extracted from the *headers* of the FITs files

In [ ]:
positions_df

For the sake of convenience

In [ ]:
pos_columns = ['RA', 'DEC', 'CRA', 'CDEC']
tsne_columns = [f'tsne_{i}' for i in range(1, 3)]

*Projections* `DataFrame` extended with positions. Notice the indexes in `proj_df` are kept.

In [ ]:
proj_df_with_positions = proj_df.join(positions_df, on=['tar file', 'source'], how='left')
assert not proj_df_with_positions.isna().any().any()
proj_df_with_positions

The above `DataFrame` is to be shared with a collaborator, who will use it to query database (notice that, actually, `positions_df` would be enough)

In [ ]:
proj_df_with_positions[pos_columns + tsne_columns].to_csv('pos_and_proj.csv', index=False)

### SIMBAD (XMATCH) database

An external catalog is used to extract some *alleged* labels associated with spatial coordinates

In [ ]:
pd.set_option('display.max_columns', None)

# catalog_file = pathlib.Path('Cross_Simbad_r3.5.csv')
catalog_file = base_dir / 'experiments' / 'Cross_Simbad_r3.5.csv'
assert catalog_file.exists()
catalog_df = pd.read_csv(catalog_file)
catalog_df

Some columns of interest

In [ ]:
catalog_columns_of_interest = ['main_type', 'other_types', 'main_id']

They are turned into *categorical*

In [ ]:
# for c in catalog_columns_of_interest:
#     catalog_df[c] = pd.Categorical(catalog_df[c])

The number of *records* in the catalog is much larger than that of requested positions (*windows*), i.e., **each query returned several matches**.

In [ ]:
len(catalog_df), len(proj_df_with_positions)

In particular, the number of *matches* for every position is

In [ ]:
groupby_catalog = catalog_df.groupby(pos_columns)
groupby_catalog.size()

Out of every group, we keep only the first element (in the given order)

In [ ]:
first_match_catalog_df = groupby_catalog.first()
first_match_catalog_df

After picking the first match for every position, the number of *matched* positions (see the above number of rows) is lower than that of *requested* positions. Hence, the DB didn't return any result for some requested positions.

In [ ]:
len(positions_df)

Data used for training is matched against the catalog (indexes are still consistent with those in `proj_df`, and hence also with those in `ds`)

In [ ]:
match_columns_of_interes = catalog_columns_of_interest + pos_columns
ext_proj_df = pd.merge(
    proj_df_with_positions,
    first_match_catalog_df.reset_index()[match_columns_of_interes], # a few columns in the catalog DataFrame
    on=pos_columns, how='left'
)
ext_proj_df

Two new columns are added
- one just combines the *tar* and *FIT* file, which univocally determines the signal,
- the other yields the number of window *within* every (*tar* file, *FIT* file) combination

In [ ]:
ext_proj_df['# window'] = ext_proj_df.groupby(['tar file', 'source']).cumcount()
ext_proj_df['signal'] = ext_proj_df['tar file'].str.replace('.tar', '') + '/' + ext_proj_df['source'].str.replace('.FTZ', '')
ext_proj_df

We make sure all the windows within a given signal all have the same `main_type`

In [ ]:
for group_name, group_df in ext_proj_df.groupby('signal'):
    assert len(group_df['main_type'].unique()) == 1

Number of *unique* signals

In [ ]:
ext_proj_df['signal'].nunique()

The number of windows for each signal

In [ ]:
ext_proj_df.groupby('signal').size()

Distinct *types* of objects

In [ ]:
ext_proj_df['main_type'].unique()

Number os windows that lack a `main_type`

In [ ]:
main_type_is_na = ext_proj_df['main_type'].isna()
main_type_is_na.sum()

Only the *identified* light curves are kept

In [ ]:
# ext_proj_df.dropna(subset=['main_type'], inplace=True)

A column encompassing the whole latent space vector for every window.

In [ ]:
# ext_proj_df['z'] = ext_proj_df.filter(like='z_').values.tolist()

Resulting `DataFrame` is saved

In [ ]:
analysis_file = model_dir / 'analysis.parquet'
ext_proj_df.to_parquet(analysis_file)

### Javi's paper

The clustering (either *in projected space* or *in z*) to be analyzed

In [ ]:
# cluster = cluster_in_z
cluster = cluster_in_projected_space

In [ ]:
ext_proj_df.loc[i_javi, tsne_columns + ['set', cluster, 'main_type']]

From [this issue](https://github.com/vega/altair/pull/3119)

In [ ]:
# hover_data=['index', 'source', cluster]
# brush = alt.selection_interval(name="interval")
# alt.data_transformers.enable("vegafusion")
# # alt_fig = alt.Chart(proj_df.reset_index()).mark_point(opacity=0.5).encode(
# alt_fig = alt.Chart(ext_proj_df.reset_index()).mark_point(opacity=0.5).encode(
#     x='tsne_1',
#     y='tsne_2',
#     # color='tar file',
#     color='main_type',
#     # shape='set',
#     tooltip=hover_data
# ).properties(
#     width=1200,
#     height=800
# ).add_params(brush).interactive()
# # jchart = alt.JupyterChart(alt_fig)
# # jchart
# alt_fig


A `DataFrame` with the points in the selection

In [ ]:
# selection = (proj_df['tsne_1'] > jchart.selections.interval.value['tsne_1'][0])\
#  & (proj_df['tsne_1'] < jchart.selections.interval.value['tsne_1'][1])\
# & (proj_df['tsne_2'] > jchart.selections.interval.value['tsne_2'][0])\
# & (proj_df['tsne_2'] < jchart.selections.interval.value['tsne_2'][1])
# selection_df = proj_df[selection]
# selection_df.head()

In [ ]:
# selection_df[cluster].value_counts()

Empty `main_type`s are set to `UNKNOWN`

In [ ]:
ext_proj_df['main_type'].fillna('UNKNOWN', inplace=True)

A few `main_type`s are selected

In [ ]:
# selected_main_type = ext_proj_df['main_type'].isin(['OrionV*', 'Low-Mass*', 'LowMassXBin', 'TTauri*'])
# fig = px.scatter(ext_proj_df[selected_main_type].reset_index(), x='tsne_1', y='tsne_2', color='main_type', hover_data=on_plot_hover_columns, opacity=0.5, width=1_200, height=800)
# fig.show()

In [ ]:
on_plot_hover_columns = ['index', 'source', cluster, 'main_type']
on_plot_hover_columns

In [ ]:
tsne_plot = trapallada.analysis.TSNE(ext_proj_df, tsne_columns, full_signals_dir, on_plot_hover_columns)
fig = tsne_plot.make_fig()
tsne_plot.highlight_windows_of_interest()
fig.show()

In [ ]:
# fig = px.scatter(ext_proj_df.reset_index(), x='tsne_1', y='tsne_2', color='main_type', hover_data=on_plot_hover_columns, opacity=0.5, width=1_200, height=800)
# # fig = px.scatter(ext_proj_df.reset_index(), x='tsne_1', y='tsne_2', color='signal', hover_data=on_plot_hover_columns, opacity=0.5, width=1_200, height=800)
# # fig = px.scatter(ext_proj_df.reset_index(), x='tsne_1', y='tsne_2', color='cluster in z', hover_data=on_plot_hover_columns, opacity=0.5, width=1_200, height=800)
# # fig = px.scatter(ext_proj_df.reset_index(), x='tsne_1', y='tsne_2', color=cluster_in_projected_space , hover_data=on_plot_hover_columns, opacity=0.5, width=1_200, height=800)
# fig.add_trace(go.Scatter(
#     x=ext_proj_df.loc[i_javi, 'tsne_1'].tolist(), y=proj_df.loc[i_javi, 'tsne_2'].tolist(), mode='markers', marker=dict(size=20, symbol='star', color='blue')
# ))
# fig.show()

The signals associated with some *manually-selected* points (i.e. windows) of interest are plotted.

In [ ]:
# i_study = [30460]
# for i in i_study:
#     display(trapallada.analysis.predict_and_plot(model, ds, i))

In [ ]:
# ext_proj_df.loc[i_study]

Some signals that do **not** belong to the main cluster (*allegedly* noise).

In [ ]:
trapallada.analysis.predict_and_plot(model, ds, i_minority[30])

In [ ]:
ext_proj_df.loc[2656][tsne_columns]

## 3D

The latent space is projected into a 3D space

In [ ]:
# proj3D_df = trapallada.util.make_projections_df(ds, fulldataset_dl, model.encoder, n_projected_components=3)
# proj3D_df.loc[train_ds[:][0], 'set'] = 'training'
# proj3D_df.loc[validation_ds[:][0], 'set'] = 'validation'
# proj3D_df.head()

In [ ]:
# proj3D_df.loc[i_javi].filter(like='tsne_')

In [ ]:
# fig = px.scatter_3d(proj3D_df.reset_index(), x='tsne_1', y='tsne_2', z='tsne_3', hover_data=['index'], color='tar file', opacity=0.5, width=1_200, height=800)
# fig.add_trace(go.Scatter3d(
#     x=proj3D_df.loc[i_javi, 'tsne_1'].tolist(), y=proj3D_df.loc[i_javi, 'tsne_2'].tolist(), z=proj3D_df.loc[i_javi, 'tsne_3'].tolist(), mode='markers', marker=dict(size=12, symbol='diamond', color='blue')
# ))
# fig.show()